In [17]:
import sys
sys.version

'3.12.12 (main, Mar  3 2026, 18:22:04) [Clang 17.0.0 (clang-1700.0.13.3)]'

In [18]:
import pandas as pd

df = pd.read_csv("../data/sample_pnl.csv")
df.head()

,pnl
0,0.015
1,-0.020
2,0.005
3,-0.010
4,0.012


In [19]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sample_pnl.csv")
df["date"] = pd.date_range("2026-01-01", periods=len(df), freq="D")  #날짜 컬럼 생성
df.head()

,pnl,date
0,0.015,2026-01-01
1,-0.020,2026-01-02
2,0.005,2026-01-03
3,-0.010,2026-01-04
4,0.012,2026-01-05


In [20]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
df = df.set_index("date")

pnl = df["pnl"]         #pandas Series
pnl.head()

date
2026-01-01    0.015
2026-01-02   -0.020
2026-01-03    0.005
2026-01-04   -0.010
2026-01-05    0.012
Name: pnl, dtype: float64

In [21]:
pnl_3day_mean = pnl.rolling(window=3).mean()
pnl_3day_mean

date
2026-01-01             NaN
2026-01-02             NaN
2026-01-03   -2.891206e-19
2026-01-04   -8.333333e-03
2026-01-05    2.333333e-03
2026-01-06   -9.333333e-03
2026-01-07   -3.333333e-03
2026-01-08   -9.333333e-03
2026-01-09    2.000000e-03
2026-01-10   -6.666667e-03
Name: pnl, dtype: float64

In [22]:
roll = pnl.rolling(3).mean()
roll_clean = roll.dropna()      #NaN이 있는 행 삭제
roll_clean

date
2026-01-03   -2.891206e-19
2026-01-04   -8.333333e-03
2026-01-05    2.333333e-03
2026-01-06   -9.333333e-03
2026-01-07   -3.333333e-03
2026-01-08   -9.333333e-03
2026-01-09    2.000000e-03
2026-01-10   -6.666667e-03
Name: pnl, dtype: float64

In [23]:
# alignment가 안 맞아서 생기는 문제와 맞춰서 해결하는 방법
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sample_pnl.csv")
df["date"] = pd.date_range("2026-01-01", periods=len(df), freq="D")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()

pnl = df["pnl"]     # pandas Series (DatetimeIndex)
pnl


date
2026-01-01    0.015
2026-01-02   -0.020
2026-01-03    0.005
2026-01-04   -0.010
2026-01-05    0.012
2026-01-06   -0.030
2026-01-07    0.008
2026-01-08   -0.006
2026-01-09    0.004
2026-01-10   -0.018
Name: pnl, dtype: float64

In [ ]:
alpha = 0.99
window = 3

# rolling VaR(관측치 기반) 비슷하게: loss quantile
loss = -pnl
var_roll = loss.rolling(window).quantile(alpha)  # index는 유지, 앞부분 NaN
var_roll_drop = var_roll.dropna()

# 흔한 실수: numpy로 바꾸면 인덱스 정보 사라짐 (아래에는 길이가 달라서 Error)
#viol_bad = loss.to_numpy() > var_roll_drop.to_numpy()

#print("len(loss:)", len(loss))
#print("len(var_roll_drop):", len(var_roll_drop))
#print("viol_bad length", len(viol_bad))
#print("viol_bad:", viol_bad)

ValueError: operands could not be broadcast together with shapes (10,) (8,) 

In [31]:
viol_good1 = (loss > var_roll).dropna()
print(viol_good1)

date
2026-01-01    False
2026-01-02    False
2026-01-03    False
2026-01-04    False
2026-01-05    False
2026-01-06     True
2026-01-07    False
2026-01-08    False
2026-01-09    False
2026-01-10     True
Name: pnl, dtype: bool


In [32]:
aligned = pd.concat([loss.rename("loss"), var_roll.rename("var")], axis=1).dropna()
viol_good2 = aligned["loss"] > aligned["var"]


print("aligned rows:", len(aligned))
print("viol_good length:", len(viol_good2))
print(viol_good2)

aligned rows: 8
viol_good length: 8
date
2026-01-03    False
2026-01-04    False
2026-01-05    False
2026-01-06     True
2026-01-07    False
2026-01-08    False
2026-01-09    False
2026-01-10     True
dtype: bool


In [14]:
import pandas as pd

df = pd.read_csv("../data/sample_pnl.csv")
df["date"] = pd.date_range("2026-01-01", periods=len(df), freq="D")
df.head()

,pnl,date
0,0.015,2026-01-01
1,-0.020,2026-01-02
2,0.005,2026-01-03
3,-0.010,2026-01-04
4,0.012,2026-01-05


In [15]:
df_before = df.copy()

df_after = df.set_index("date").sort_index()

print("=== BEFORE (date is a column) ===")
display(df_before.head())

print("=== AFTER (date is index) ===")
display(df_after.head())

print("Before index type:", type(df_before.index))
print("After index type:", type(df_after.index))

=== BEFORE (date is a column) ===


,pnl,date
0,0.015,2026-01-01
1,-0.020,2026-01-02
2,0.005,2026-01-03
3,-0.010,2026-01-04
4,0.012,2026-01-05


=== AFTER (date is index) ===


,pnl
date,
2026-01-01,0.015
2026-01-02,-0.020
2026-01-03,0.005
2026-01-04,-0.010
2026-01-05,0.012


Before index type: <class 'pandas.RangeIndex'>
After index type: <class 'pandas.DatetimeIndex'>


In [ ]:
# max vs quantile(0.99)
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sample_pnl.csv")
df["date"] = pd.date_range("2026-01-01", periods=len(df), freq="D")
df = df.set_index("date").sort_index()
pnl = df["pnl"]
pnl

date
2026-01-01    0.015
2026-01-02   -0.020
2026-01-03    0.005
2026-01-04   -0.010
2026-01-05    0.012
2026-01-06   -0.030
2026-01-07    0.008
2026-01-08   -0.006
2026-01-09    0.004
2026-01-10   -0.018
Name: pnl, dtype: float64

In [34]:
alpha = 0.99
window = 3

pnl_window = pnl.iloc[-window:]
loss_window = -pnl_window  # Series 그대로(인덱스 유지)

print("=== pnl last 3 ===")
print(pnl_window)

print("\n=== loss last 3 ===")
print(loss_window)

# 정렬된 loss 값 확인(직관용)
loss_sorted = np.sort(loss_window.to_numpy())
print("\nloss sorted:", loss_sorted)

max_loss = float(loss_window.max())
q_linear = float(np.quantile(loss_window.to_numpy(), alpha, method="linear"))

print(f"\nmax(loss)        = {max_loss:.6f}")
print(f"quantile(loss,{alpha}, linear) = {q_linear:.6f}")
print(f"difference (q - max) = {q_linear - max_loss:.12f}")

=== pnl last 3 ===
date
2026-01-08   -0.006
2026-01-09    0.004
2026-01-10   -0.018
Name: pnl, dtype: float64

=== loss last 3 ===
date
2026-01-08    0.006
2026-01-09   -0.004
2026-01-10    0.018
Name: pnl, dtype: float64

loss sorted: [-0.004  0.006  0.018]

max(loss)        = 0.018000
quantile(loss,0.99, linear) = 0.017760
difference (q - max) = -0.000240000000


In [35]:
methods = ["linear", "lower", "higher", "nearest", "midpoint"]
arr = loss_window.to_numpy()

print("=== quantile methods comparison ===")
for m in methods:
    q = np.quantile(arr, alpha, method=m)
    print(f"{m:8s}: {q:.6f}")

=== quantile methods comparison ===
linear  : 0.017760
lower   : 0.006000
higher  : 0.018000
nearest : 0.018000
midpoint: 0.012000
